# OpenCV — Mining Equipment Visual Damage

This is the **separate image track** of the capstone. It does not use or link to TiNA sensor records. Its current purpose is to check a local image collection, inspect labels and provenance, and preview OpenCV preprocessing. **No damage classifier or future-failure predictor has been trained here.**

An image can show existing damage. To predict damage *before it appears*, we would need repeated, dated images of the same machine plus a later damage or repair outcome.

## Image sources to check (not yet training data)

- [Mine-shovel tooth study](https://pmc.ncbi.nlm.nih.gov/articles/PMC8629673/): real shovel-camera images for missing-tooth detection. The paper says raw data are available from the authors; it does not provide a public image download. Request the images and usage permission before using them.
- [Public bucket-tooth dataset](https://universe.roboflow.com/pangtu/bucket-teeth-6mc53): downloadable excavator-component images under CC BY 4.0. Its listed classes are `Excavator_claw` and `Excavator_head`, **not damage labels**; use only as a component-localization prototype, not a damage result. Mine provenance is not documented on the dataset page.
- [Missing-tooth image collection](https://universe.roboflow.com/buckettooth-detection/bucket-tooth-missing-model): lists `tooth` and `tooth_ok`, but has no released dataset version or source description. Do not rely on it for a reproducible mining-damage claim until those gaps are resolved.

A link is not a local dataset. No images are downloaded by this notebook, and none of these sources currently supports a verified trained mining-damage model here.

## Local data and labels

Put permitted images under `data/raw/opencv/` in this repository. The folder is ignored by Git. Optionally add `data/raw/opencv/image_labels.csv` with one row per image and these columns:

`image_path,equipment_type,component,damage_label,source,usage_permission,equipment_id,capture_time`

`image_path` is relative to `data/raw/opencv/`, for example `shovel/001.jpg`. The first six fields identify the image and its current-condition label. `equipment_id` and `capture_time` can be blank for a static-image study, but they and later outcomes are necessary for a future-damage study. Keep the source and permission for every image. Do not mix multiple equipment types into one performance number without evaluating each type separately.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

working_dir = Path.cwd().resolve()
project_root = working_dir if (working_dir / 'README.md').is_file() else working_dir.parent
if not (project_root / 'README.md').is_file():
    raise RuntimeError('Open this notebook from the project root or its notebooks folder.')

image_root = project_root / 'data' / 'raw' / 'opencv'
manifest_path = image_root / 'image_labels.csv'
print(f'Image folder: {image_root}')
print(f'OpenCV version: {cv2.__version__}')

In [ ]:
image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
image_files = (
    sorted(path for path in image_root.rglob('*')
           if path.is_file() and path.suffix.lower() in image_suffixes)
    if image_root.is_dir() else []
)
print(f'Local images found: {len(image_files)}')
if not image_files:
    print('No images yet. Obtain an approved, labelled mining-equipment image source first.')

if manifest_path.is_file():
    labels = pd.read_csv(manifest_path)
    required_columns = {
        'image_path', 'equipment_type', 'component',
        'damage_label', 'source', 'usage_permission'
    }
    missing_columns = sorted(required_columns - set(labels.columns))
    print(f'Manifest rows: {len(labels)}')
    if missing_columns:
        print(f'Missing required columns: {missing_columns}')
    else:
        missing_paths = labels['image_path'].fillna('').map(
            lambda name: not (image_root / str(name)).is_file()
        )
        print(f'Rows with missing image files: {int(missing_paths.sum())}')
        print('Equipment types:')
        print(labels['equipment_type'].value_counts(dropna=False).to_string())
        print('Damage labels:')
        print(labels['damage_label'].value_counts(dropna=False).to_string())
else:
    print('No label manifest yet; image previews alone cannot train or evaluate a model.')

In [ ]:
if image_files:
    selected_path = image_files[0]
    image_bgr = cv2.imread(str(selected_path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError(f'OpenCV could not read {selected_path}')

    scale = min(1.0, 900 / max(image_bgr.shape[:2]))
    preview_bgr = cv2.resize(image_bgr, None, fx=scale, fy=scale)
    original_rgb = cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2LAB)
    lightness, channel_a, channel_b = cv2.split(lab)
    equalized = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lightness)
    enhanced_bgr = cv2.cvtColor(
        cv2.merge((equalized, channel_a, channel_b)), cv2.COLOR_LAB2BGR
    )
    enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)
    edges = cv2.Canny(cv2.cvtColor(preview_bgr, cv2.COLOR_BGR2GRAY), 80, 160)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for axis, image, title, cmap in zip(
        axes, (original_rgb, enhanced_rgb, edges),
        ('Original', 'Contrast preview', 'Edges preview'),
        (None, None, 'gray')
    ):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis('off')
    fig.suptitle(str(selected_path.relative_to(image_root)))
    plt.tight_layout()
    plt.show()
else:
    print('Preview skipped: no local images. Contrast and edges are not damage predictions.')

## Before training a model

1. Verify mining-machine provenance, usage permission, and independently checked damage labels.
2. Count labelled examples per equipment type and damage class; choose a realistic first target.
3. Keep photos of the same equipment or near-duplicate video frames in the same split. If dated follow-up exists, evaluate on later time periods.
4. Add a simple baseline and report per-equipment recall, precision, false alarms, and examples of errors.
5. Call the result *future-damage prediction* only if the target is a later damage/repair event; otherwise call it *visual damage detection*.

This notebook deliberately stops before training because no approved mining-equipment image dataset has been selected.